# 3. Diet problem (Week 2 tutorial Q6b; the slides call it Q5b): maximise attractiveness, which is unbounded

Lecture: *Week 3 - Linear Programming 2*, slides 13–14.

*Original script: `5 Source Files/Python/Week2_question_5b_pyomo.py`. Runs in the browser with HiGHS instead of GLPK; the model and answers are unchanged.*

In [ ]:
# Run this cell first.
# In the browser (JupyterLite) it installs the extra packages into this tab;
# in Colab or Anaconda it installs them with pip (already installed = no change).
%pip install pyomo highspy

import sys
if sys.platform == "emscripten":  # only in the browser (JupyterLite): adapt Pyomo to it
    import contextlib, pyomo.common.tee, pyomo.common.formatting as _fmt
    # 1) browsers have no threads: skip Pyomo's threaded capture of solver output
    pyomo.common.tee.capture_output = lambda *args, **kwargs: contextlib.nullcontext()
    for _m in ("pyomo.contrib.appsi.solvers.highs", "pyomo.contrib.solver.solvers.highs"):
        if _m in sys.modules:
            sys.modules[_m].capture_output = pyomo.common.tee.capture_output
    # 2) the notebook's output returns None from write(); model.pprint()/display() need a number
    if not getattr(_fmt.StreamIndenter, "_browser_patched", False):
        class _Counted:
            def __init__(self, s): self._s = s
            def write(self, data): self._s.write(data); return len(data)
            def __getattr__(self, name): return getattr(self._s, name)
        def _init(self, ostream, indent=" " * 4, _orig=_fmt.StreamIndenter.__init__):
            _orig(self, ostream, indent)
            if not isinstance(self.target_os, _Counted):
                object.__setattr__(self, "target_os", _Counted(self.target_os))
        _fmt.StreamIndenter.__init__ = _init
        _fmt.StreamIndenter._browser_patched = True

In [ ]:
from pyomo.environ import *

In [ ]:
# Define the attractiveness values as a list
a = [6, 1, 5, 5, 6, 4, 10, 10, 6, 7]

In [ ]:
# Define the vitamin and fat values as lists
vita = [18, 8, 22, 6, 0, 0, 8, 4, 0, 12]
vitb = [4, 6, 10, 12, 14, 0, 0, 0, 6, 20]
vitc = [0, 22, 0, 0, 0, 20, 0, 8, 10, 0]
vitd = [6, 14, 0, 6, 10, 8, 6, 0, 12, 4]
fat = [0, 0, 8, 0, 0, 6, 20, 40, 0, 0]

no_vars = len(a)

In [ ]:
# Create a concrete model
model = ConcreteModel()

In [ ]:
# Define the decision variables
model.x = Var(range(no_vars), domain=NonNegativeReals)

In [ ]:
# Define the objective function
model.obj = Objective(expr=sum(a[i] * model.x[i] for i in range(no_vars)), sense=maximize)

In [ ]:
# Define constraints
model.vita_min = Constraint(expr=sum(vita[i] * model.x[i] for i in range(no_vars)) >= 100)
model.vitb_min = Constraint(expr=sum(vitb[i] * model.x[i] for i in range(no_vars)) >= 120)
model.vitc_min = Constraint(expr=sum(vitc[i] * model.x[i] for i in range(no_vars)) >= 200)
model.vitd_min = Constraint(expr=sum(vitd[i] * model.x[i] for i in range(no_vars)) >= 80)
model.fat_max = Constraint(expr=sum(fat[i] * model.x[i] for i in range(no_vars)) <= 300)

In [ ]:
# Solve the optimization problem
solver = SolverFactory('appsi_highs')
results = solver.solve(model, tee=True, load_solutions=False)  # Set tee=True to display solver output
# This version of the diet problem is UNBOUNDED: nothing stops the model adding more
# fat-free food (see the "Fix" slide and Q5c),
# so report the solver's verdict instead of loading a solution.
print('Termination condition:', results.solver.termination_condition)

# # Check the solver status and print the results
# if results.solver.status == SolverStatus.ok:
#     if results.solver.termination_condition == TerminationCondition.optimal:
#         print("Optimal solution found.")
#         print("Objective value:", model.obj())
#         print("Decision variables:")
#         for i in range(no_vars):
#             print(f"x[{i}] = {model.x[i].value}")
#     else:
#         print("Solver terminated.")
# else:
#     print("Solver did not find a solution. Status:", results.solver.status)